In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# simulator

In [ ]:
"""
Adaptive-network SIR epidemic simulator.

This module simulates an SIR (Susceptible-Infected-Recovered) epidemic
spreading on a network that evolves over time. The key idea is that
susceptible individuals can "rewire" their connections to avoid infected
neighbors, which couples the disease dynamics with the network topology.

The model proceeds in discrete time steps, each with three phases:
  1. Infection: infected nodes transmit the disease to susceptible neighbors
  2. Recovery: infected nodes recover (and become immune)
  3. Rewiring: susceptible nodes break links with infected neighbors and
     form new connections elsewhere

Reference: Gross et al. (2006), "Epidemic dynamics on an adaptive network",
Physical Review Letters, 96(20), 208701.
"""

import numpy as np


def simulate(beta, gamma, rho, N=200, p_edge=0.05, n_infected0=5, T=200, rng=None):
    """Run one replicate of the adaptive-network SIR model.

    Parameters
    ----------
    beta : float in [0, 1]
        Transmission probability. At each time step, each S-I edge
        transmits the infection independently with probability beta.
        Higher beta means the disease spreads faster.
    gamma : float in [0, 1]
        Recovery probability. At each time step, each infected node
        recovers independently with probability gamma.
        Higher gamma means shorter infectious period (on average 1/gamma steps).
    rho : float in [0, 1]
        Rewiring probability. At each time step, each S-I edge is
        rewired independently with probability rho. The susceptible
        node drops the link to its infected neighbor and connects to
        a randomly chosen new node instead.
        Higher rho means more active social distancing behavior.
    N : int, default=200
        Number of nodes (individuals) in the network.
    p_edge : float, default=0.05
        Probability of an edge between any two nodes in the initial
        Erdos-Renyi random graph. Expected initial degree is (N-1)*p_edge.
        With N=200 and p_edge=0.05, the expected degree is about 10.
    n_infected0 : int, default=5
        Number of nodes infected at time t=0. These are chosen
        uniformly at random (without replacement) from all N nodes.
    T : int, default=200
        Number of discrete time steps to simulate.
    rng : numpy.random.Generator or None
        Random number generator for reproducibility. If None, a new
        generator is created with an arbitrary seed. Pass
        np.random.default_rng(seed) for reproducible runs.

    Returns
    -------
    infected_fraction : np.ndarray, shape (T+1,)
        Fraction of the population that is infected at each time step,
        from t=0 to t=T. Values are in [0, 1].
    rewire_counts : np.ndarray, shape (T+1,)
        Number of successful rewiring events at each time step.
        Always 0 at t=0 (no rewiring before the simulation starts).
    degree_histogram : np.ndarray, shape (31,)
        Histogram of node degrees at the final time step t=T.
        degree_histogram[d] = number of nodes with degree d, for d=0..29.
        degree_histogram[30] counts all nodes with degree >= 30.
    """
    if rng is None:
        rng = np.random.default_rng()

    # =====================================================================
    # STEP 0: Build the initial contact network as an Erdos-Renyi graph.
    #
    # We represent the network as an adjacency list using Python sets.
    # neighbors[i] is the set of node indices connected to node i.
    # Sets allow O(1) lookups for "is j a neighbor of i?" and efficient
    # add/remove operations, which is important for the rewiring step.
    #
    # For each pair (i, j) with i < j, we add an edge with probability
    # p_edge. This produces an undirected graph (if i is connected to j,
    # then j is also connected to i).
    # =====================================================================
    neighbors = [set() for _ in range(N)]
    for i in range(N):
        for j in range(i + 1, N):
            if rng.random() < p_edge:
                neighbors[i].add(j)
                neighbors[j].add(i)

    # =====================================================================
    # Initialize the health state of each node.
    #
    # We encode states as integers:
    #   0 = Susceptible (S): can catch the disease
    #   1 = Infected (I):    currently infectious
    #   2 = Recovered (R):   immune, cannot be infected again
    #
    # At t=0, we pick n_infected0 nodes uniformly at random to be infected.
    # All other nodes start as susceptible.
    # =====================================================================
    state = np.zeros(N, dtype=np.int8)
    initial_infected = rng.choice(N, size=n_infected0, replace=False)
    state[initial_infected] = 1

    # Arrays to record the summary statistics at each time step
    infected_fraction = np.zeros(T + 1)
    rewire_counts = np.zeros(T + 1, dtype=np.int64)
    infected_fraction[0] = np.sum(state == 1) / N

    # =================================================================
    # Main simulation loop: iterate over T discrete time steps.
    # Each time step has three phases applied in order:
    #   Phase 1: Infection (S -> I transitions)
    #   Phase 2: Recovery  (I -> R transitions)
    #   Phase 3: Rewiring  (network topology changes)
    # =================================================================
    for t in range(1, T + 1):

        # =============================================================
        # PHASE 1: INFECTION (synchronous update)
        #
        # For every infected node i, look at each of its neighbors j.
        # If j is susceptible (state 0), the infection transmits with
        # probability beta.
        #
        # Important: we use synchronous (parallel) updating. We first
        # collect ALL new infections in a set, then apply them all at
        # once. This prevents "chain infections" within a single step
        # (where a newly infected node immediately infects its own
        # neighbors in the same step).
        # =============================================================
        new_infections = set()
        infected_nodes = np.where(state == 1)[0]

        for i in infected_nodes:
            for j in neighbors[i]:
                if state[j] == 0:  # j is susceptible
                    if rng.random() < beta:
                        new_infections.add(j)

        # Apply all new infections at once (synchronous update)
        for j in new_infections:
            state[j] = 1

        # =============================================================
        # PHASE 2: RECOVERY
        #
        # Each currently infected node (including those just infected
        # in Phase 1) recovers independently with probability gamma.
        # Recovery is permanent: recovered nodes move to state 2 (R)
        # and can never be infected again.
        #
        # We recompute the infected set to include newly infected nodes.
        # =============================================================
        infected_nodes = np.where(state == 1)[0]
        for i in infected_nodes:
            if rng.random() < gamma:
                state[i] = 2

        # =============================================================
        # PHASE 3: NETWORK REWIRING (adaptive behavior)
        #
        # This is what makes the model "adaptive": the network structure
        # changes in response to the disease.
        #
        # We look at all edges between a susceptible node (S) and an
        # infected node (I), called "S-I edges". For each such edge,
        # with probability rho, the susceptible node:
        #   1. Drops the connection to its infected neighbor
        #   2. Forms a new connection to a randomly chosen other node
        #      (that it is not already connected to)
        #
        # This models social distancing: susceptible individuals
        # actively avoid infected contacts.
        # =============================================================
        rewire_count = 0

        # First, collect all S-I edges. We iterate over susceptible
        # nodes and check their neighbors for infected ones.
        si_edges = []
        for i in range(N):
            if state[i] == 0:  # node i is susceptible
                for j in neighbors[i]:
                    if state[j] == 1:  # neighbor j is infected
                        si_edges.append((i, j))

        # Process each S-I edge for potential rewiring
        for s_node, i_node in si_edges:
            if rng.random() < rho:
                # Check that this edge still exists. An earlier rewiring
                # in this same loop may have already removed it (since
                # rewiring can affect shared neighborhoods).
                if i_node not in neighbors[s_node]:
                    continue

                # Remove the S-I edge (break the link in both directions)
                neighbors[s_node].discard(i_node)
                neighbors[i_node].discard(s_node)

                # Find all valid candidates for a new connection:
                # any node that is not s_node itself and not already
                # a neighbor of s_node. Note that the new partner can
                # be in any state (S, I, or R).
                candidates = []
                for k in range(N):
                    if k != s_node and k not in neighbors[s_node]:
                        candidates.append(k)

                # If there is at least one valid candidate, pick one
                # uniformly at random and create the new edge
                if candidates:
                    new_partner = rng.choice(candidates)
                    neighbors[s_node].add(new_partner)
                    neighbors[new_partner].add(s_node)
                    rewire_count += 1

        # Record summary statistics for this time step
        infected_fraction[t] = np.sum(state == 1) / N
        rewire_counts[t] = rewire_count

    # =====================================================================
    # Compute the degree histogram at the final time step.
    #
    # The degree of a node is its number of connections (neighbors).
    # We bin degrees from 0 to 29 individually, and lump all degrees >= 30
    # into a single bin (index 30). This gives a fixed-size output array
    # of shape (31,) regardless of the actual degree distribution.
    # =====================================================================
    degree_histogram = np.zeros(31, dtype=np.int64)
    for i in range(N):
        deg = min(len(neighbors[i]), 30)
        degree_histogram[deg] += 1

    return infected_fraction, rewire_counts, degree_histogram

# data

In [ ]:
df_infected = pd.read_csv('/Users/millan/Desktop/Simulation/infected_timeseries.csv')
df_rewiring = pd.read_csv('/Users/millan/Desktop/Simulation/rewiring_timeseries.csv')
df_degree = pd.read_csv('/Users/millan/Desktop/Simulation/final_degree_histograms.csv')

# set C re-run

In [ ]:

N = 200_000

rng = np.random.default_rng(123) # different seed from before

Beta_prior = rng.uniform(0.05, 0.50, size = N)
Gamma_prior = rng.uniform(0.02, 0.20, size = N)
Rho_prior = rng.uniform(0.0, 0.8, size = N) 

# Combine the three 1D arrays into one (N x 3) array of parameter triples
theta_generated_array = np.column_stack((Beta_prior, Gamma_prior, Rho_prior))

In [ ]:
#set C is set A + set B + set C

rewiring_mean_by_time = df_rewiring.groupby("time")["rewire_count"].mean().values

peak_rewiring_obs = np.max(rewiring_mean_by_time)
time_to_peak_rewiring_obs = np.argmax(rewiring_mean_by_time)
total_rewiring_obs = np.sum(rewiring_mean_by_time)


infected_mean_by_time = df_infected.groupby("time")["infected_fraction"].mean().values #as multiple replicates at same time points, here are taking the mean at each time t to get a single trajectory.

peak_infected_obs = np.max(infected_mean_by_time)
time_to_peak_obs = np.argmax(infected_mean_by_time)
area_under_curve_obs = np.sum(infected_mean_by_time)


df_degree_wide = df_degree.pivot(
    index="replicate_id",
    columns="degree",
    values="count"
).fillna(0)

degree_hist_obs = df_degree_wide.mean(axis=0).values

degree_probs_obs = degree_hist_obs / np.sum(degree_hist_obs)
degrees = np.arange(len(degree_probs_obs))

mean_degree_obs = np.sum(degrees * degree_probs_obs)
var_degree_obs = np.sum((degrees - mean_degree_obs)**2 * degree_probs_obs)
high_degree_mass_obs = np.sum(degree_probs_obs[15:])

S_obs_ABC = np.array([peak_infected_obs, time_to_peak_obs, area_under_curve_obs, peak_rewiring_obs, time_to_peak_rewiring_obs, total_rewiring_obs, mean_degree_obs, var_degree_obs, high_degree_mass_obs]) #is our observed summary so is shape (9, 0) (one vector)
print(S_obs_ABC)


#SUMMARY OF SIMULATED DATA

def compute_summary_ABC(infected_fraction, rewiring_count, degree_histogram): #from the simulator

    #from set A
    peak_infected = np.max(infected_fraction)
    
    time_to_peak_infected = np.argmax(infected_fraction)
    
    auc_infected = np.sum(infected_fraction)

    #from set B
    peak_rewiring = np.max(rewiring_count)

    time_to_peak_rewiring = np.argmax(rewiring_count)

    total_rewiring = np.sum(rewiring_count)

    #from set C
    degree_probs = degree_histogram / np.sum(degree_histogram)
    degrees = np.arange(len(degree_probs))


    mean_degree = np.sum(degrees * degree_probs)
    var_degree = np.sum((degrees - mean_degree)**2 * degree_probs)
    high_degree_mass = np.sum(degree_probs[15:])
    
    return np.array([peak_infected, time_to_peak_infected, auc_infected, peak_rewiring, time_to_peak_rewiring, total_rewiring, mean_degree, var_degree, high_degree_mass])



all_summaries_ABC = [] #here is where store the summaries for all N=2000 simulations

for i in range(N):
    beta, gamma, rho = theta_generated_array[i] #is the index of parameter triple using for simulation i

    infected_fraction_sim, rewire_counts_sim, degree_histogram_sim = simulate(beta, gamma, rho, rng = rng) #run the simulation for this parameter triple and get the infected fraction time series, rewiring count time series and final degree histogram

    s_sim_ABC = compute_summary_ABC(infected_fraction_sim, rewire_counts_sim, degree_histogram_sim) #compute the summary for this simulation and store it in s_sim_AB
    all_summaries_ABC.append(s_sim_ABC) #append the summary for this simulation to the list of all summaries

all_summaries_ABC = np.array(all_summaries_ABC)  #should be (N, 9) where N is number of simulations and 9 is number of summary stats in set ABC

print(df_degree_wide.sum(axis=1).head())

In [ ]:
# standard deviation of each summary statistic across all simulations
std_ABC = np.std(all_summaries_ABC, axis=0)

# avoid divide-by-zero just in case one summary has zero spread
std_ABC[std_ABC == 0] = 1e-8

# normalized Euclidean distance between each simulated summary and observed summary
distances_ABC = np.linalg.norm((all_summaries_ABC - S_obs_ABC) / std_ABC, axis=1)


#here at 5% quantile
epsilon_ABC_5 = np.quantile(distances_ABC, 0.05) #here accepting closest 5% of simulations
accepted_indices_ABC_5 = np.where(distances_ABC <= epsilon_ABC_5)[0]
accepted_thetas_ABC_5 = theta_generated_array[accepted_indices_ABC_5]

print("Epsilon:", epsilon_ABC_5)
print("Number accepted:", len(accepted_indices_ABC_5))

#here at 2% quantile
epsilon_ABC_2 = np.quantile(distances_ABC, 0.02) #here accepting closest 2% of simulations
accepted_indices_ABC_2 = np.where(distances_ABC <= epsilon_ABC_2)[0]
accepted_thetas_ABC_2 = theta_generated_array[accepted_indices_ABC_2]


print("Epsilon:", epsilon_ABC_2)
print("Number accepted:", len(accepted_indices_ABC_2))

#here at 1% quantile
epsilon_ABC_1 = np.quantile(distances_ABC, 0.01) #here accepting closest 1% of simulations
accepted_indices_ABC_1 = np.where(distances_ABC <= epsilon_ABC_1)[0]
accepted_thetas_ABC_1 = theta_generated_array[accepted_indices_ABC_1]


print("Epsilon:", epsilon_ABC_1)
print("Number accepted:", len(accepted_indices_ABC_1))

In [ ]:
beta_accepted_ABC_5 = accepted_thetas_ABC_5[:, 0]
gamma_accepted_ABC_5 = accepted_thetas_ABC_5[:, 1]
rho_accepted_ABC_5 = accepted_thetas_ABC_5[:, 2]

beta_accepted_ABC_2 = accepted_thetas_ABC_2[:, 0]
gamma_accepted_ABC_2 = accepted_thetas_ABC_2[:, 1]
rho_accepted_ABC_2 = accepted_thetas_ABC_2[:, 2]

beta_accepted_ABC_1 = accepted_thetas_ABC_1[:, 0]
gamma_accepted_ABC_1 = accepted_thetas_ABC_1[:, 1]
rho_accepted_ABC_1 = accepted_thetas_ABC_1[:, 2]


def summary_stats(samples):
    mean = np.mean(samples)
    median = np.median(samples)
    lower = np.quantile(samples, 0.025)
    upper = np.quantile(samples, 0.975)
    
    return mean, median, lower, upper



#for below first plot values I get from diff quantiles, see which one is best and use that to compute actuall predictions for values
mean_beta_ABC, median_beta_ABC, low_beta_ABC, high_beta_ABC = summary_stats(beta_accepted_ABC_2)

print("Beta:")
print("Mean:", mean_beta_ABC)
print("Median:", median_beta_ABC)
print("95% CI:", (low_beta_ABC, high_beta_ABC))


mean_gamma_ABC, median_gamma_ABC, low_gamma_ABC, high_gamma_ABC = summary_stats(gamma_accepted_ABC_2)

print("Gamma:")
print("Mean:", mean_gamma_ABC)
print("Median:", median_gamma_ABC)
print("95% CI:", (low_gamma_ABC, high_gamma_ABC))

mean_rho_ABC, median_rho_ABC, low_rho_ABC, high_rho_ABC = summary_stats(rho_accepted_ABC_2)

print("Rho:")
print("Mean:", mean_rho_ABC)
print("Median:", median_rho_ABC)
print("95% CI:", (low_rho_ABC, high_rho_ABC))

In [ ]:
np.save("/Users/millan/Desktop/accepted_thetas_ABC_run2_5.npy", accepted_thetas_ABC_5)
np.save("/Users/millan/Desktop/accepted_thetas_ABC_run2_2.npy", accepted_thetas_ABC_2)
np.save("/Users/millan/Desktop/accepted_thetas_ABC_run2_1.npy", accepted_thetas_ABC_1)

print("Saved to Desktop.")

# for alternative distance 

In [ ]:
# standard deviation of each summary statistic across all simulations
std_ABC_L1 = np.std(all_summaries_ABC, axis=0)

# avoid divide-by-zero just in case one summary has zero spread
std_ABC_L1[std_ABC_L1 == 0] = 1e-8

# standardised L1 distance between each simulated summary and observed summary
distances_ABC_L1 = np.sum(np.abs((all_summaries_ABC - S_obs_ABC) / std_ABC_L1), axis=1)


# here at 5% quantile
epsilon_ABC_L1_5 = np.quantile(distances_ABC_L1, 0.05)  # accepting closest 5% of simulations
accepted_indices_ABC_L1_5 = np.where(distances_ABC_L1 <= epsilon_ABC_L1_5)[0]
accepted_thetas_ABC_L1_5 = theta_generated_array[accepted_indices_ABC_L1_5]

print("L1 Epsilon:", epsilon_ABC_L1_5)
print("L1 Number accepted:", len(accepted_indices_ABC_L1_5))

# here at 2% quantile
epsilon_ABC_L1_2 = np.quantile(distances_ABC_L1, 0.02)  # accepting closest 2% of simulations
accepted_indices_ABC_L1_2 = np.where(distances_ABC_L1 <= epsilon_ABC_L1_2)[0]
accepted_thetas_ABC_L1_2 = theta_generated_array[accepted_indices_ABC_L1_2]

print("L1 Epsilon:", epsilon_ABC_L1_2)
print("L1 Number accepted:", len(accepted_indices_ABC_L1_2))

# here at 1% quantile
epsilon_ABC_L1_1 = np.quantile(distances_ABC_L1, 0.01)  # accepting closest 1% of simulations
accepted_indices_ABC_L1_1 = np.where(distances_ABC_L1 <= epsilon_ABC_L1_1)[0]
accepted_thetas_ABC_L1_1 = theta_generated_array[accepted_indices_ABC_L1_1]

print("L1 Epsilon:", epsilon_ABC_L1_1)
print("L1 Number accepted:", len(accepted_indices_ABC_L1_1))


beta_accepted_ABC_L1_5 = accepted_thetas_ABC_L1_5[:, 0]
gamma_accepted_ABC_L1_5 = accepted_thetas_ABC_L1_5[:, 1]
rho_accepted_ABC_L1_5 = accepted_thetas_ABC_L1_5[:, 2]

beta_accepted_ABC_L1_2 = accepted_thetas_ABC_L1_2[:, 0]
gamma_accepted_ABC_L1_2 = accepted_thetas_ABC_L1_2[:, 1]
rho_accepted_ABC_L1_2 = accepted_thetas_ABC_L1_2[:, 2]

beta_accepted_ABC_L1_1 = accepted_thetas_ABC_L1_1[:, 0]
gamma_accepted_ABC_L1_1 = accepted_thetas_ABC_L1_1[:, 1]
rho_accepted_ABC_L1_1 = accepted_thetas_ABC_L1_1[:, 2]


def summary_stats(samples):
    mean = np.mean(samples)
    median = np.median(samples)
    lower = np.quantile(samples, 0.025)
    upper = np.quantile(samples, 0.975)
    
    return mean, median, lower, upper


# for below first plot values I get from diff quantiles, see which one is best and use that to compute actual predictions for values
mean_beta_ABC_L1, median_beta_ABC_L1, low_beta_ABC_L1, high_beta_ABC_L1 = summary_stats(beta_accepted_ABC_L1_2)

print("Beta (L1):")
print("Mean:", mean_beta_ABC_L1)
print("Median:", median_beta_ABC_L1)
print("95% CI:", (low_beta_ABC_L1, high_beta_ABC_L1))


mean_gamma_ABC_L1, median_gamma_ABC_L1, low_gamma_ABC_L1, high_gamma_ABC_L1 = summary_stats(gamma_accepted_ABC_L1_2)

print("Gamma (L1):")
print("Mean:", mean_gamma_ABC_L1)
print("Median:", median_gamma_ABC_L1)
print("95% CI:", (low_gamma_ABC_L1, high_gamma_ABC_L1))


mean_rho_ABC_L1, median_rho_ABC_L1, low_rho_ABC_L1, high_rho_ABC_L1 = summary_stats(rho_accepted_ABC_L1_2)

print("Rho (L1):")
print("Mean:", mean_rho_ABC_L1)
print("Median:", median_rho_ABC_L1)
print("95% CI:", (low_rho_ABC_L1, high_rho_ABC_L1))

# coeffiecient scatter plot for Set C ABC

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# load if needed
# accepted = np.load("accepted_thetas_ABC_run2_2.npy")

accepted = accepted_thetas_ABC_2  # if still in memory

beta  = accepted[:, 0]
gamma = accepted[:, 1]
rho   = accepted[:, 2]

plt.figure(figsize=(15,5))

# β vs γ
plt.subplot(1, 3, 1)
plt.scatter(beta, gamma, alpha=0.25, s=8)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\gamma$')
plt.title(r'$\beta$ vs $\gamma$')
plt.xlim(0.05, 0.50)
plt.ylim(0.02, 0.20)

# β vs ρ
plt.subplot(1, 3, 2)
plt.scatter(beta, rho, alpha=0.25, s=8)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\rho$')
plt.title(r'$\beta$ vs $\rho$')
plt.xlim(0.05, 0.50)
plt.ylim(0.0, 0.80)

# γ vs ρ  ⭐ important one
plt.subplot(1, 3, 3)
plt.scatter(gamma, rho, alpha=0.25, s=8)
plt.xlabel(r'$\gamma$')
plt.ylabel(r'$\rho$')
plt.title(r'$\gamma$ vs $\rho$')
plt.xlim(0.02, 0.20)
plt.ylim(0.0, 0.80)

plt.tight_layout()
plt.savefig("/Users/millan/Desktop/ABC_SetC_Joint_2percent.png",dpi=300, bbox_inches="tight")
plt.show()

# regression adjusted second run

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

def regression_adjust_abc(accepted_indices, all_summaries, theta_array, s_obs):
    """
    Perform Beaumont-style regression adjustment on accepted ABC samples.

    Parameters
    ----------
    accepted_indices : array-like
        Indices of accepted simulations.
    all_summaries : np.ndarray, shape (N, d)
        Summary statistics for all simulations.
    theta_array : np.ndarray, shape (N, p)
        Parameter draws for all simulations.
    s_obs : np.ndarray, shape (d,)
        Observed summary vector.

    Returns
    -------
    accepted_thetas : np.ndarray, shape (n_acc, p)
        Original accepted parameter draws.
    adjusted_thetas : np.ndarray, shape (n_acc, p)
        Regression-adjusted parameter draws.
    """
    # accepted summaries and parameters
    s_acc = all_summaries[accepted_indices]         # shape (n_acc, d)
    theta_acc = theta_array[accepted_indices]       # shape (n_acc, 3)

    # difference between accepted simulated summaries and observed summaries
    X = s_acc - s_obs                               # shape (n_acc, d)

    # array to store adjusted parameters
    theta_adj = np.zeros_like(theta_acc)

    # fit one regression per parameter
    for j in range(theta_acc.shape[1]):
        y = theta_acc[:, j]

        reg = LinearRegression()
        reg.fit(X, y)

        # predicted correction term for each accepted point
        correction = reg.predict(X) - reg.intercept_

        # Beaumont adjustment: theta* = theta - b^T (s - s_obs)
        theta_adj[:, j] = y - correction

    return theta_acc, theta_adj


# ---------------------------
# Run regression adjustment at 5%, 2%, 1%
# ---------------------------

accepted_thetas_ABC_5, adjusted_thetas_ABC_5 = regression_adjust_abc(
    accepted_indices_ABC_5, all_summaries_ABC, theta_generated_array, S_obs_ABC
)

accepted_thetas_ABC_2, adjusted_thetas_ABC_2 = regression_adjust_abc(
    accepted_indices_ABC_2, all_summaries_ABC, theta_generated_array, S_obs_ABC
)

accepted_thetas_ABC_1, adjusted_thetas_ABC_1 = regression_adjust_abc(
    accepted_indices_ABC_1, all_summaries_ABC, theta_generated_array, S_obs_ABC
)

In [ ]:
#as adjusted regression can produce values outside of prior range, here am clipping them to be within the original prior bounds

adjusted_thetas_ABC_5[:, 0] = np.clip(adjusted_thetas_ABC_5[:, 0], 0.05, 0.50)  # beta
adjusted_thetas_ABC_5[:, 1] = np.clip(adjusted_thetas_ABC_5[:, 1], 0.02, 0.20)  # gamma
adjusted_thetas_ABC_5[:, 2] = np.clip(adjusted_thetas_ABC_5[:, 2], 0.00, 0.80)  # rho

adjusted_thetas_ABC_2[:, 0] = np.clip(adjusted_thetas_ABC_2[:, 0], 0.05, 0.50)
adjusted_thetas_ABC_2[:, 1] = np.clip(adjusted_thetas_ABC_2[:, 1], 0.02, 0.20)
adjusted_thetas_ABC_2[:, 2] = np.clip(adjusted_thetas_ABC_2[:, 2], 0.00, 0.80)

adjusted_thetas_ABC_1[:, 0] = np.clip(adjusted_thetas_ABC_1[:, 0], 0.05, 0.50)
adjusted_thetas_ABC_1[:, 1] = np.clip(adjusted_thetas_ABC_1[:, 1], 0.02, 0.20)
adjusted_thetas_ABC_1[:, 2] = np.clip(adjusted_thetas_ABC_1[:, 2], 0.00, 0.80)

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde

def posterior_stats(samples, label):
    param_names = ["beta", "gamma", "rho"]
    
    print(f"\n--- {label} ---")
    
    for j, name in enumerate(param_names):
        vals = samples[:, j]
        
        # mean & median
        mean_val = np.mean(vals)
        median_val = np.median(vals)
        
        # 95% CI
        ci_lower, ci_upper = np.quantile(vals, [0.025, 0.975])
        
        # mode via KDE
        kde = gaussian_kde(vals)
        x_grid = np.linspace(vals.min(), vals.max(), 1000)
        mode_val = x_grid[np.argmax(kde(x_grid))]
        
        print(
            f"{name}: mean={mean_val:.3f}, median={median_val:.3f}, "
            f"mode={mode_val:.3f}, 95% CI=({ci_lower:.3f}, {ci_upper:.3f})"
        )

In [ ]:
posterior_stats(adjusted_thetas_ABC_2, "Adjusted ABC (2%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# use adjusted 2% samples
accepted = adjusted_thetas_ABC_2

beta  = accepted[:, 0]
gamma = accepted[:, 1]
rho   = accepted[:, 2]

plt.figure(figsize=(15,5))

# β vs γ
plt.subplot(1, 3, 1)
plt.scatter(beta, gamma, alpha=0.25, s=8)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\gamma$')
plt.title(r'$\beta$ vs $\gamma$ (Regression)')
plt.xlim(0.05, 0.50)
plt.ylim(0.02, 0.20)

# β vs ρ
plt.subplot(1, 3, 2)
plt.scatter(beta, rho, alpha=0.25, s=8)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\rho$')
plt.title(r'$\beta$ vs $\rho$ (Regression)')
plt.xlim(0.05, 0.50)
plt.ylim(0.0, 0.80)

# γ vs ρ
plt.subplot(1, 3, 3)
plt.scatter(gamma, rho, alpha=0.25, s=8)
plt.xlabel(r'$\gamma$')
plt.ylabel(r'$\rho$')
plt.title(r'$\gamma$ vs $\rho$ (Regression)')
plt.xlim(0.02, 0.20)
plt.ylim(0.0, 0.80)

plt.tight_layout()

# save (use Desktop or safe path)
plt.savefig("/Users/millan/Desktop/Regression_SetC_Joint_2percent.png",
            dpi=300, bbox_inches="tight")

plt.show()

# SMC - ABC Run 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

In [ ]:
import numpy as np

np.random.seed(456)   # fixed seed for SMC reproducibility

# prior bounds
prior_bounds = np.array([
    [0.05, 0.50],   # beta
    [0.02, 0.20],   # gamma
    [0.00, 0.80]    # rho
])

def sample_from_prior():  # for first population sample theta from prior
    beta = np.random.uniform(0.05, 0.50)
    gamma = np.random.uniform(0.02, 0.20)
    rho = np.random.uniform(0.00, 0.80)
    return np.array([beta, gamma, rho])

def in_prior_support(theta, bounds=prior_bounds):  # check sample within bounds
    for j in range(len(theta)):
        if theta[j] < bounds[j, 0] or theta[j] > bounds[j, 1]:
            return False
    return True

def prior_density(theta, bounds=prior_bounds):  # compute prior density
    if not in_prior_support(theta, bounds):
        return 0.0
    volume = np.prod(bounds[:, 1] - bounds[:, 0])
    return 1.0 / volume

In [ ]:
def abc_distance(summary_sim, summary_obs, std_vec):
    return np.linalg.norm((summary_sim - summary_obs) / std_vec) 

In [ ]:
def weighted_covariance(particles, weights): #computing spread of particles, taking into account their weights
    """
    particles: shape (N, d)
    weights: shape (N,)
    """

    weights = weights / np.sum(weights)
    
    mean = np.average(particles, axis=0, weights=weights)
    centered = particles - mean
    cov = np.zeros((particles.shape[1], particles.shape[1]))
    
    for i in range(len(weights)):
        cov += weights[i] * np.outer(centered[i], centered[i])
    
    # normalize by 1 - sum(w^2) for weighted covariance stability
    denom = 1.0 - np.sum(weights**2)
    if denom > 1e-12:
        cov /= denom
    else:
        cov += 1e-8 * np.eye(particles.shape[1])
    
    return cov

In [ ]:
def simulate_and_summarise(theta): #takes the theta and runs the simulation and returns the summary stats, this is what we will call in ABC-SMC for each particle
    beta, gamma, rho = theta
    infected_fraction_sim, rewire_counts_sim, degree_histogram_sim = simulate(beta, gamma, rho)
    s_sim = compute_summary_ABC(infected_fraction_sim, rewire_counts_sim, degree_histogram_sim)
    return s_sim

In [ ]:
def initialise_population(N_particles, epsilon, s_obs, std_vec): #initialise first population of particles by sampling from prior and accepting those that are within epsilon distance of observed summary. First SMC population
    particles = []
    summaries = []
    distances = []
    n_simulations = 0
    
    while len(particles) < N_particles:
        theta = sample_from_prior()
        s_sim = simulate_and_summarise(theta)
        d = abc_distance(s_sim, s_obs, std_vec)
        n_simulations += 1
        
        if d <= epsilon:
            particles.append(theta)
            summaries.append(s_sim)
            distances.append(d)
    
    particles = np.array(particles)
    summaries = np.array(summaries)
    distances = np.array(distances)
    weights = np.ones(N_particles) / N_particles
    
    return particles, summaries, distances, weights, n_simulations

In [ ]:
def smc_iteration(prev_particles, prev_weights, N_particles, epsilon, s_obs, std_vec, bounds=prior_bounds): #given previous population of particles and weights, generate new population by resampling, perturbing and accepting those within epsilon distance of observed summary.
    d = prev_particles.shape[1]
    
    # adaptive kernel covariance: 2 x weighted covariance
    cov_prev = weighted_covariance(prev_particles, prev_weights)
    kernel_cov = 2.0 * cov_prev + 1e-8 * np.eye(d)
    
    new_particles = []
    new_summaries = []
    new_distances = []
    n_simulations = 0
    
    while len(new_particles) < N_particles:
        # resample parent index according to weights
        idx = np.random.choice(len(prev_particles), p=prev_weights)
        parent = prev_particles[idx]
        
        # perturb
        theta_star = np.random.multivariate_normal(mean=parent, cov=kernel_cov)
        
        # reject immediately if outside prior support
        if not in_prior_support(theta_star, bounds):
            continue
        
        # simulate and test against current epsilon
        s_sim = simulate_and_summarise(theta_star)
        d_sim = abc_distance(s_sim, s_obs, std_vec)
        n_simulations += 1
        
        if d_sim <= epsilon:
            new_particles.append(theta_star)
            new_summaries.append(s_sim)
            new_distances.append(d_sim)
    
    new_particles = np.array(new_particles)
    new_summaries = np.array(new_summaries)
    new_distances = np.array(new_distances)
    
    # importance weights, compute importance weights for new population
    new_weights = np.zeros(N_particles)
    
    for i in range(N_particles):
        theta_i = new_particles[i]
        
        numerator = prior_density(theta_i, bounds)
        
        denominator = 0.0
        for j in range(len(prev_particles)):
            denominator += prev_weights[j] * multivariate_normal.pdf(
                theta_i, mean=prev_particles[j], cov=kernel_cov
            )
        
        new_weights[i] = numerator / max(denominator, 1e-300)
    
    # normalize
    new_weights /= np.sum(new_weights)
    
    return new_particles, new_summaries, new_distances, new_weights, kernel_cov, n_simulations

In [ ]:
def run_smc_abc(N_particles, epsilon_schedule, s_obs, std_vec): #main function to run ABC-SMC, iterating through the epsilon schedule and storing results for each population
    all_populations = []
    all_weights = []
    all_summaries = []
    all_distances = []
    all_kernel_covs = []
    sim_counts = []
    
    print(f"Initialising population at epsilon = {epsilon_schedule[0]:.4f}")
    particles, summaries, distances, weights, n_sim = initialise_population(
        N_particles, epsilon_schedule[0], s_obs, std_vec
    )
    
    all_populations.append(particles)
    all_weights.append(weights)
    all_summaries.append(summaries)
    all_distances.append(distances)
    all_kernel_covs.append(None)
    sim_counts.append(n_sim)
    
    print(f"Population 1 complete: {N_particles} particles accepted using {n_sim} simulations")
    
    for t in range(1, len(epsilon_schedule)):
        eps = epsilon_schedule[t]
        print(f"Running population {t+1} at epsilon = {eps:.4f}")
        
        particles, summaries, distances, weights, kernel_cov, n_sim = smc_iteration(
            all_populations[-1],
            all_weights[-1],
            N_particles,
            eps,
            s_obs,
            std_vec
        )
        
        all_populations.append(particles)
        all_weights.append(weights)
        all_summaries.append(summaries)
        all_distances.append(distances)
        all_kernel_covs.append(kernel_cov)
        sim_counts.append(n_sim)
        
        ess = 1.0 / np.sum(weights**2)
        print(f"Population {t+1} complete: ESS = {ess:.1f}, simulations used = {n_sim}")
    
    return {
        "particles": all_populations,
        "weights": all_weights,
        "summaries": all_summaries,
        "distances": all_distances,
        "kernel_covs": all_kernel_covs,
        "sim_counts": sim_counts,
        "epsilons": epsilon_schedule
    }

In [ ]:
epsilon_schedule = [                     # here defining the epsilon schedule for ABC-SMC, starting from a looser tolerance and tightening it down to 1% quantile
    np.quantile(distances_ABC, 0.10),
    np.quantile(distances_ABC, 0.05),
    np.quantile(distances_ABC, 0.02),
    np.quantile(distances_ABC, 0.01)
]

print(epsilon_schedule)

In [ ]:
N_particles = 2_000 # number of particles per population, can adjust based on computational resources. **CHANGE LATER**

smc_results = run_smc_abc(
    N_particles=N_particles,
    epsilon_schedule=epsilon_schedule,
    s_obs=S_obs_ABC,
    std_vec=std_ABC
)

In [ ]:
final_particles = smc_results["particles"][-1] #get the final population of particles from the last iteration of ABC-SMC
final_weights = smc_results["weights"][-1]
final_epsilon = smc_results["epsilons"][-1]

print("Final epsilon:", final_epsilon)
print("Final population shape:", final_particles.shape) 
print("Final ESS:", 1.0 / np.sum(final_weights**2))

In [ ]:

def weighted_quantile(values, quantiles, sample_weight=None):
    values = np.asarray(values)
    quantiles = np.atleast_1d(quantiles)

    if sample_weight is None:
        sample_weight = np.ones(len(values))
    sample_weight = np.asarray(sample_weight)

    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]

    cumulative_weight = np.cumsum(sample_weight)
    cumulative_weight = cumulative_weight / cumulative_weight[-1]

    return np.interp(quantiles, cumulative_weight, values)


def summarise_weighted_posterior(samples, weights, label):
    param_names = ["beta", "gamma", "rho"]
    
    
    weights = weights / np.sum(weights)

    print(f"\n--- {label} ---")

    for j, name in enumerate(param_names):
        vals = samples[:, j]

        mean_val = np.sum(weights * vals)
        median_val = weighted_quantile(vals, 0.5, weights)[0]
        ci_lower, ci_upper = weighted_quantile(vals, [0.025, 0.975], weights)

        print(
            f"{name}: mean={mean_val:.3f}, "
            f"median={median_val:.3f}, "
            f"95% CI=({ci_lower:.3f}, {ci_upper:.3f})"
        )

summarise_weighted_posterior(
    final_particles[:, :3],
    final_weights,
    "SMC-ABC final"
)

In [ ]:
def plot_weighted_smc_posteriors_p2(samples, weights, label="SMC-ABC posterior", save_path="SMC_ABC_final.png"):
    param_names = [r"$\beta$", r"$\gamma$", r"$\rho$"]
    
    # prior ranges
    param_ranges = [
        (0.05, 0.50),  # beta
        (0.02, 0.20),  # gamma
        (0.00, 0.80)   # rho
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for j, ax in enumerate(axes):
        ax.hist(
            samples[:, j],
            bins=30,
            weights=weights,
            density=True,
            alpha=0.7,
            range=param_ranges[j]  # ensures bins span full prior
        )
        
        ax.set_xlim(param_ranges[j])  # force axis to full prior range
        
        ax.set_title(f"{param_names[j]} ({label})")
        ax.set_xlabel(param_names[j])
        ax.set_ylabel("Density")
    
    plt.tight_layout()

    

    plt.show()

plot_weighted_smc_posteriors_p2(final_particles, final_weights, label="SMC-ABC final")

In [ ]:
print(type(smc_results))
print(len(smc_results))

for i, x in enumerate(smc_results):
    print(i, type(x))

In [ ]:
import numpy as np

np.save("/Users/millan/Desktop/smc_final_particles.npy", final_particles)
np.save("/Users/millan/Desktop/smc_final_weights.npy", final_weights)

print("Saved SMC final particles and weights.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# use your final SMC particles
particles = final_particles   # already loaded or from smc_results

beta  = particles[:, 0]
gamma = particles[:, 1]
rho   = particles[:, 2]

plt.figure(figsize=(15,5))

# β vs γ
plt.subplot(1, 3, 1)
plt.scatter(beta, gamma, alpha=0.25, s=8)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\gamma$')
plt.title(r'$\beta$ vs $\gamma$ (SMC-ABC)')
plt.xlim(0.05, 0.50)
plt.ylim(0.02, 0.20)

# β vs ρ
plt.subplot(1, 3, 2)
plt.scatter(beta, rho, alpha=0.25, s=8)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\rho$')
plt.title(r'$\beta$ vs $\rho$ (SMC-ABC)')
plt.xlim(0.05, 0.50)
plt.ylim(0.0, 0.80)

# γ vs ρ
plt.subplot(1, 3, 3)
plt.scatter(gamma, rho, alpha=0.25, s=8)
plt.xlabel(r'$\gamma$')
plt.ylabel(r'$\rho$')
plt.title(r'$\gamma$ vs $\rho$ (SMC-ABC)')
plt.xlim(0.02, 0.20)
plt.ylim(0.0, 0.80)

plt.tight_layout()

# save safely (use Desktop path)
plt.savefig("/Users/millan/Desktop/SMC_SetC_Joint.png",
            dpi=300, bbox_inches="tight")

plt.show()

# Posterior predictive Overlays

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Observed data prep

infected_mean_by_time = df_infected.groupby("time")["infected_fraction"].mean().values
rewiring_mean_by_time = df_rewiring.groupby("time")["rewire_count"].mean().values

df_degree_wide = df_degree.pivot(
    index="replicate_id",
    columns="degree",
    values="count"
).fillna(0)

degree_hist_obs = df_degree_wide.mean(axis=0).values
degree_probs_obs = degree_hist_obs / np.sum(degree_hist_obs)

time_points = np.arange(len(infected_mean_by_time))
degree_bins = np.arange(len(degree_probs_obs))



# Simulate posterior predictive draws

def simulate_replicates_from_posterior(theta_samples, n_draws=50):
    """
    theta_samples: array of shape (N, 3)
    n_draws: number of posterior draws to simulate
    """

    
    idx = np.random.choice(len(theta_samples), size=n_draws, replace=True)
    theta_draws = theta_samples[idx]

    infected_sims = []
    rewiring_sims = []
    degree_sims = []

    for theta in theta_draws:
        beta, gamma, rho = theta
        infected_fraction_sim, rewire_counts_sim, degree_histogram_sim = simulate(
            beta, gamma, rho
        )

        infected_sims.append(infected_fraction_sim)
        rewiring_sims.append(rewire_counts_sim)
        degree_sims.append(degree_histogram_sim)

    infected_sims = np.array(infected_sims)
    rewiring_sims = np.array(rewiring_sims)
    degree_sims = np.array(degree_sims)

    return infected_sims, rewiring_sims, degree_sims



# Posterior predictive plots

def posterior_predictive_plots(theta_samples, method_name, save_prefix=None, n_draws=50):
    infected_sims, rewiring_sims, degree_sims = simulate_replicates_from_posterior(
        theta_samples, n_draws=n_draws
    )

    degree_probs_sims = degree_sims / degree_sims.sum(axis=1, keepdims=True)

    # Infection trajectories
    plt.figure(figsize=(8, 5))
    for i in range(n_draws):
        plt.plot(time_points, infected_sims[i], color="gray", alpha=0.2)
    plt.plot(time_points, infected_mean_by_time, color="black", linewidth=2, label="Observed")
    plt.xlabel("Time")
    plt.ylabel("Infected fraction")
    plt.title(f"Posterior predictive: infection trajectory ({method_name})")
    plt.legend()
    plt.tight_layout()
    if save_prefix is not None:
        plt.savefig(f"/Users/millan/Desktop/{save_prefix}_infected.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Rewiring trajectories
    plt.figure(figsize=(8, 5))
    for i in range(n_draws):
        plt.plot(time_points, rewiring_sims[i], color="gray", alpha=0.2)
    plt.plot(time_points, rewiring_mean_by_time, color="black", linewidth=2, label="Observed")
    plt.xlabel("Time")
    plt.ylabel("Rewiring count")
    plt.title(f"Posterior predictive: rewiring trajectory ({method_name})")
    plt.legend()
    plt.tight_layout()
    if save_prefix is not None:
        plt.savefig(f"/Users/millan/Desktop/{save_prefix}_rewiring.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Degree distribution
    plt.figure(figsize=(8, 5))
    for i in range(n_draws):
        plt.plot(degree_bins, degree_probs_sims[i], color="gray", alpha=0.2)
    plt.plot(degree_bins, degree_probs_obs, color="black", linewidth=2, label="Observed")
    plt.xlabel("Degree")
    plt.ylabel("Probability")
    plt.title(f"Posterior predictive: degree distribution ({method_name})")
    plt.legend()
    plt.tight_layout()
    if save_prefix is not None:
        plt.savefig(f"/Users/millan/Desktop/{save_prefix}_degree.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
#for rejection ABC

posterior_predictive_plots(
    accepted_thetas_ABC_2,
    method_name="Rejection ABC (Set C, 2%)",
    save_prefix="ABC_postpred",
    n_draws=50
)

In [ ]:
# for regression-adjusted ABC

posterior_predictive_plots(
    adjusted_thetas_ABC_2,
    method_name="Regression-adjusted ABC (Set C, 2%)",
    save_prefix="ABC_Regression_postpred",
    n_draws=50
)

In [ ]:
#for SMC-ABC

posterior_predictive_plots(
    final_particles[:, :3],
    method_name="SMC-ABC",
    save_prefix="SMC_ABC_postpred",
    n_draws=50
)

# Set C ABC Synthetic Truth Recovery Experiment

In [ ]:

theta_star = (0.20, 0.10, 0.30)
beta_star, gamma_star, rho_star = theta_star

infected_syn, rewiring_syn, degree_syn = simulate(beta_star, gamma_star, rho_star)

# treat this synthetic dataset as the "observed" data
infected_mean_by_time = infected_syn
rewiring_mean_by_time = rewiring_syn
degree_hist_obs = degree_syn

degree_probs_obs = degree_hist_obs / np.sum(degree_hist_obs)
degrees = np.arange(len(degree_probs_obs))

# Set A summaries
peak_infected_obs = np.max(infected_mean_by_time)
time_to_peak_obs = np.argmax(infected_mean_by_time)
area_under_curve_obs = np.sum(infected_mean_by_time)

# Set B summaries
peak_rewiring_obs = np.max(rewiring_mean_by_time)
time_to_peak_rewiring_obs = np.argmax(rewiring_mean_by_time)
total_rewiring_obs = np.sum(rewiring_mean_by_time)

# Set C summaries
mean_degree_obs = np.sum(degrees * degree_probs_obs)
var_degree_obs = np.sum((degrees - mean_degree_obs) ** 2 * degree_probs_obs)
high_degree_mass_obs = np.sum(degree_probs_obs[15:])

S_obs_ABC = np.array([
    peak_infected_obs,
    time_to_peak_obs,
    area_under_curve_obs,
    peak_rewiring_obs,
    time_to_peak_rewiring_obs,
    total_rewiring_obs,
    mean_degree_obs,
    var_degree_obs,
    high_degree_mass_obs
])

print("Synthetic truth theta* =", theta_star)
print("Synthetic observed summary vector:")
print(S_obs_ABC)

In [ ]:

# draw from prior 

N = 200_000

Beta_prior = np.random.uniform(0.05, 0.50, size=N)
Gamma_prior = np.random.uniform(0.02, 0.20, size=N)
Rho_prior = np.random.uniform(0.0, 0.8, size=N)

theta_generated_array = np.column_stack((Beta_prior, Gamma_prior, Rho_prior))


In [ ]:
# Summary statistic function for Set C

def compute_summary_ABC(infected_fraction, rewiring_count, degree_histogram):
    # Set A
    peak_infected = np.max(infected_fraction)
    time_to_peak_infected = np.argmax(infected_fraction)
    auc_infected = np.sum(infected_fraction)

    # Set B
    peak_rewiring = np.max(rewiring_count)
    time_to_peak_rewiring = np.argmax(rewiring_count)
    total_rewiring = np.sum(rewiring_count)

    # Set C
    degree_probs = degree_histogram / np.sum(degree_histogram)
    degrees = np.arange(len(degree_probs))

    mean_degree = np.sum(degrees * degree_probs)
    var_degree = np.sum((degrees - mean_degree) ** 2 * degree_probs)
    high_degree_mass = np.sum(degree_probs[15:])

    return np.array([
        peak_infected,
        time_to_peak_infected,
        auc_infected,
        peak_rewiring,
        time_to_peak_rewiring,
        total_rewiring,
        mean_degree,
        var_degree,
        high_degree_mass
    ])


# Simulate summaries under prior

N = 100_000

all_summaries_ABC = []

for i in range(N):
    beta, gamma, rho = theta_generated_array[i]

    infected_fraction_sim, rewire_counts_sim, degree_histogram_sim = simulate(beta, gamma, rho)

    s_sim_ABC = compute_summary_ABC(
        infected_fraction_sim,
        rewire_counts_sim,
        degree_histogram_sim
    )
    all_summaries_ABC.append(s_sim_ABC)

all_summaries_ABC = np.array(all_summaries_ABC)
print("all_summaries_ABC shape:", all_summaries_ABC.shape)



In [ ]:

#Standardised Euclidean distances

std_ABC = np.std(all_summaries_ABC, axis=0)
std_ABC[std_ABC == 0] = 1e-8

distances_ABC = np.linalg.norm((all_summaries_ABC - S_obs_ABC) / std_ABC, axis=1)



#Accept top 2%

epsilon_ABC_2 = np.quantile(distances_ABC, 0.02)
accepted_indices_ABC_2 = np.where(distances_ABC <= epsilon_ABC_2)[0]

accepted_thetas_ABC_2 = theta_generated_array[accepted_indices_ABC_2]
accepted_summaries_ABC_2 = all_summaries_ABC[accepted_indices_ABC_2]

print("Epsilon (2%):", epsilon_ABC_2)
print("Number accepted:", len(accepted_indices_ABC_2))


#Plain Set C posterior summaries

def summary_stats(samples):
    mean = np.mean(samples)
    median = np.median(samples)
    lower = np.quantile(samples, 0.025)
    upper = np.quantile(samples, 0.975)
    return mean, median, lower, upper

beta_accepted_ABC_2 = accepted_thetas_ABC_2[:, 0]
gamma_accepted_ABC_2 = accepted_thetas_ABC_2[:, 1]
rho_accepted_ABC_2 = accepted_thetas_ABC_2[:, 2]

mean_beta_ABC, median_beta_ABC, low_beta_ABC, high_beta_ABC = summary_stats(beta_accepted_ABC_2)
mean_gamma_ABC, median_gamma_ABC, low_gamma_ABC, high_gamma_ABC = summary_stats(gamma_accepted_ABC_2)
mean_rho_ABC, median_rho_ABC, low_rho_ABC, high_rho_ABC = summary_stats(rho_accepted_ABC_2)

print("\n--- Plain Set C ABC posterior (2%) ---")
print("Beta:")
print("Mean:", mean_beta_ABC)
print("Median:", median_beta_ABC)
print("95% CI:", (low_beta_ABC, high_beta_ABC))
print("Recovered?", low_beta_ABC <= beta_star <= high_beta_ABC)

print("\nGamma:")
print("Mean:", mean_gamma_ABC)
print("Median:", median_gamma_ABC)
print("95% CI:", (low_gamma_ABC, high_gamma_ABC))
print("Recovered?", low_gamma_ABC <= gamma_star <= high_gamma_ABC)

print("\nRho:")
print("Mean:", mean_rho_ABC)
print("Median:", median_rho_ABC)
print("95% CI:", (low_rho_ABC, high_rho_ABC))
print("Recovered?", low_rho_ABC <= rho_star <= high_rho_ABC)

In [ ]:

#Regression adjustment


from sklearn.linear_model import LinearRegression
import numpy as np

def regression_adjustment(theta_accepted, summaries_accepted, s_obs):
    adjusted_thetas = np.zeros_like(theta_accepted)

    # discrepancies between accepted simulated summaries and observed summaries
    X = summaries_accepted - s_obs   # shape (n_acc, d)

    for j in range(theta_accepted.shape[1]):
        y = theta_accepted[:, j]

        reg = LinearRegression()
        reg.fit(X, y)

        # Beaumont adjustment:
        # theta_i_adj = theta_i - b_hat^T (s_i - s_obs)
        adjustment = X @ reg.coef_
        adjusted_thetas[:, j] = y - adjustment

    return adjusted_thetas


adjusted_thetas_ABC_2 = regression_adjustment(
    accepted_thetas_ABC_2,
    accepted_summaries_ABC_2,
    S_obs_ABC
)

# clip back to prior support
adjusted_thetas_ABC_2[:, 0] = np.clip(adjusted_thetas_ABC_2[:, 0], 0.05, 0.50)  # beta
adjusted_thetas_ABC_2[:, 1] = np.clip(adjusted_thetas_ABC_2[:, 1], 0.02, 0.20)  # gamma
adjusted_thetas_ABC_2[:, 2] = np.clip(adjusted_thetas_ABC_2[:, 2], 0.00, 0.80)  # rho


# Regression-adjusted posterior summaries

beta_adj = adjusted_thetas_ABC_2[:, 0]
gamma_adj = adjusted_thetas_ABC_2[:, 1]
rho_adj = adjusted_thetas_ABC_2[:, 2]

mean_beta_adj, median_beta_adj, low_beta_adj, high_beta_adj = summary_stats(beta_adj)
mean_gamma_adj, median_gamma_adj, low_gamma_adj, high_gamma_adj = summary_stats(gamma_adj)
mean_rho_adj, median_rho_adj, low_rho_adj, high_rho_adj = summary_stats(rho_adj)

print("\n--- Regression-adjusted Set C ABC posterior (2%) ---")
print("Beta:")
print("Mean:", mean_beta_adj)
print("Median:", median_beta_adj)
print("95% CI:", (low_beta_adj, high_beta_adj))
print("Recovered?", low_beta_adj <= beta_star <= high_beta_adj)

print("\nGamma:")
print("Mean:", mean_gamma_adj)
print("Median:", median_gamma_adj)
print("95% CI:", (low_gamma_adj, high_gamma_adj))
print("Recovered?", low_gamma_adj <= gamma_star <= high_gamma_adj)

print("\nRho:")
print("Mean:", mean_rho_adj)
print("Median:", median_rho_adj)
print("95% CI:", (low_rho_adj, high_rho_adj))
print("Recovered?", low_rho_adj <= rho_star <= high_rho_adj)

In [ ]:
# compact recovery table

print("\n--- Recovery summary ---")
print(f"True beta  = {beta_star:.3f} | Plain CI = ({low_beta_ABC:.3f}, {high_beta_ABC:.3f}) | Adjusted CI = ({low_beta_adj:.3f}, {high_beta_adj:.3f})")
print(f"True gamma = {gamma_star:.3f} | Plain CI = ({low_gamma_ABC:.3f}, {high_gamma_ABC:.3f}) | Adjusted CI = ({low_gamma_adj:.3f}, {high_gamma_adj:.3f})")
print(f"True rho   = {rho_star:.3f} | Plain CI = ({low_rho_ABC:.3f}, {high_rho_ABC:.3f}) | Adjusted CI = ({low_rho_adj:.3f}, {high_rho_adj:.3f})")